In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_log_error
import seaborn as sns
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt

In [ ]:

DATA_PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting" 

train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'), parse_dates=['date'])
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'), parse_dates=['date'])
oil = pd.read_csv(os.path.join(DATA_PATH, 'oil.csv'), parse_dates=['date'])
stores = pd.read_csv(os.path.join(DATA_PATH, 'stores.csv'))
holidays = pd.read_csv(os.path.join(DATA_PATH, 'holidays_events.csv'), parse_dates=['date'])
transactions = pd.read_csv(os.path.join(DATA_PATH, 'transactions.csv'), parse_dates=['date'])

In [ ]:
train['sales'] = np.log1p(train['sales'])
print(f"train: {train.shape}, test: {test.shape}")

In [ ]:
print(train.shape)
train.info()
train.head()

In [ ]:
oil.isnull().sum()

Not all calendar dates are present because the oil price isn't published every day.

Some values are missing even for existing dates.
Need to fill it with linear interpolation.

Overview of dataset

train (Core series)

* id: Unique row identifier.

* date: The timeline (target for seasonality analysis).

* store_nbr / family: The "Grain" of the forecast; defines the individual time series.

* sales: The target variable (Units sold).

* onpromotion: A proxy for external demand spikes caused by marketing.

oil (Economic proxy)

* date: Daily timestamp.

* dcoilwtico: Daily WTI oil price; used to capture the purchasing power of the Ecuador economy.

holidays_events (Calendar events)

* type / description: Categorization of events (Holidays vs. Events vs. Earthquakes).

* locale / locale_name: Geographical scope; used to filter if a holiday actually affects a specific store.

* transferred: Boolean; identifies if a holiday was moved to a different day (crucial for "true" date modeling).

stores(Meta data)

* city / state: Regional identifiers for holiday matching.

* type / cluster: Categorical groupings of stores based on size and historical similarity.

transactions(Customer traffic)

* transactions: Count of tickets issued; highly correlated with sales, used to model "foot traffic" patterns.

In [ ]:
date_tables = {
    "train": train,
    "test": test,
    "oil": oil,
    "holidays": holidays,
    "transactions": transactions,
}

for name, df in date_tables.items():
    print("min:", df["date"].min())
    print("max:", df["date"].max())
    print()

range of dates:
* train ends at 2017-08-15, test begins at 2017-08-16 -> no skips
* oil covers all period but needs some adjustment
* holidays has dates outside of train and test, need to choose specific dates
* transactions has only train dates, I will use day lags after 16 days


In [ ]:
daily_sales = train.groupby("date")["sales"].sum()

plt.figure()
daily_sales.plot(title="Total Sales Over Time")
plt.show()

train.groupby(train["date"].dt.year)["sales"].sum().plot(
    kind="bar", title="Sales by Year"
)
plt.show()

Sales are increasing towards 2017, but 2017 itseld is not a full year, cannot be compared to other years 

In [ ]:
train["month"] = train["date"].dt.month
month_sales = (
    train[train["date"].dt.year != 2017]
    .groupby("month")["sales"]
    .mean()
)
sns.barplot(x=month_sales.index, y=month_sales.values)
plt.title("Average Sales by Month (without 2017)")
plt.show()

More sales at the end of the year and week

In [ ]:
train["day_name"] = train["date"].dt.day_name()

dow_sales = train.groupby("day_name")["sales"].mean()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

sns.barplot(x=dow_sales.index, y=dow_sales.values, order=order)
plt.title("Average Sales by Day of Week")
plt.show()

In [ ]:
a = train[["store_nbr", "sales"]]
a["ind"] = 1
a["ind"] = a.groupby("store_nbr").ind.cumsum().values

a = pd.pivot(a, index = "ind", columns = "store_nbr", values = "sales").corr()

plt.figure(figsize=(20, 20))
sns.heatmap(a,
        annot=True,
        fmt='.1f',
        cmap='coolwarm',
        square=True,
        linewidths=1,
        cbar=False)
plt.title("Correlation between stores")
plt.show()

Okay this looks horrible, I will fix this graph

But we can see that most stores do not affect each other, with exceptions stores like 52, 42, 29, 22, 21, 20



In [ ]:
store_sales = train.groupby("store_nbr")["sales"].sum().sort_values(ascending=False)

store_sales.head(20).plot(kind="bar", title="Top 20 Stores by Sales")
plt.show()

train.groupby("store_nbr")["sales"].mean().hist(bins=30)
plt.title("Store Sales Distribution")
plt.show()

family_sales = train.groupby("family")["sales"].sum().sort_values(ascending=False)

family_sales.head(20).plot(kind="bar", title="Top Categories by Sales")
plt.show()

np.log1p(train["sales"]).hist(bins=50)
plt.title("Log Sales Distribution")
plt.show()

train["sales"].hist(bins=50)
plt.title("Sales Distribution")
plt.show()

train["sales"].eq(0).astype(int).hist()
plt.title("Zero Sales Distribution")
plt.show()

Type of store is linked with sales rate, can be used as a feauture 

In [ ]:
family_sales = train.groupby("family")["sales"].sum().sort_values(ascending=False)

family_sales.head(20).plot(kind="bar", title="Top Categories by Sales")
plt.show()

train["sales"].hist(bins=50)
plt.title("Sales Distribution")
plt.show()

np.log1p(train["sales"]).hist(bins=50)
plt.title("Log Sales Distribution")
plt.show()

After the log1p transformation, the distribution becomes more compact, and the model is less likely to overfit to outliers.

In [ ]:
(train["sales"] == 0).mean()

roughly 31% is zero sales, log1p works fine with zeroes


In [ ]:

zero_by_family = train.groupby("family")["sales"].apply(lambda x: (x == 0).mean())

zero_by_family.sort_values(ascending=False).head(10)

Some categories have a lot of zeroes, this means some categories have different demands

In [ ]:
daily_sales = train.groupby("date")["sales"].sum().sort_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(daily_sales, lags=60, ax=axes[0], title="ACF — summ daily sales (60 lags)")
plot_pacf(daily_sales, lags=60, ax=axes[1], title="PACF — summ daily sales (60 lags )", method="ywm")

plt.tight_layout()
plt.show()

top_store = train.groupby("store_nbr")["sales"].sum().idxmax()
top_family = train.groupby("family")["sales"].sum().idxmax()

single_ts = (
    train[(train["store_nbr"] == top_store) & (train["family"] == top_family)]
    .groupby("date")["sales"]
    .sum()
    .sort_index()
)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(single_ts, lags=60, ax=axes[0],
         title=f"ACF — Store {top_store} | {top_family}")
plot_pacf(single_ts, lags=60, ax=axes[1],
          title=f"PACF — Store {top_store} | {top_family}", method="ywm")

plt.tight_layout()
plt.show()

# significant lags for total daily sales

acf_vals = acf(daily_sales, nlags=30, alpha=0.05)
pacf_vals = pacf(daily_sales, nlags=30, alpha=0.05, method="ywm")

# aproximate significance bound 2/sqrt(n)
conf_bound = 2 / np.sqrt(len(daily_sales))

sig_acf_lags = [i for i in range(1, 31) if abs(acf_vals[0][i]) > conf_bound]
sig_pacf_lags = [i for i in range(1, 31) if abs(pacf_vals[0][i]) > conf_bound]

print(f"significant lags ACF (до 30):  {sig_acf_lags}")
print(f"significant lags PAC (до 30): {sig_pacf_lags}")
print(f"significance bound: ±{conf_bound:.4f}")

ACF: significant lags 1-30

PACF: lags 1-9 short and weekly pattern, 13-23 2-3 week pattern, 26-28 end of the month 

I can use in model: dynamics lasting for few days, weekly cycles, yearly cycles and moving average windows for stochastic noise 


In [ ]:
corr_df = (
    train.groupby("date")[["sales", "onpromotion"]]
    .mean()
    .merge(oil[["date", "dcoilwtico"]], on="date", how="left")
    .merge(
        transactions.groupby("date")["transactions"].mean().reset_index(),
        on="date", how="left"
    )
)

corr_matrix = corr_df[["sales", "onpromotion", "transactions", "dcoilwtico"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("correlation of number features")
plt.tight_layout()
plt.show()

Big correlation between sales and onpromotion, transactions, dcoilwtico

When big promo -> more sales, more customer traffic -> more sales, oil price is linked with the demand



EDA summary:

* train and test go in sequence, 2017-08-15 -> 2017-08-16.

* Sales are shifted to the right, 31% of values are 0.

* For model I wil use log1p(sales), zero rule for days with no sales 

* ACF and PACF showed significance of lags, I will some lags and rolling means

* onpromotion is linked with sales, transactions are only available in train

* store_nbr, store_type, cluster and family are important differebt stores, different demands

* and hollidays for consideration new year, cinco de mayo etc.

* I will also use fourier sin_day, cos_day for cycles 



calendar: day_of_week, month, year, is_weekend, day_of_year, week_of_year, date_index.

Fourier: sin_day, cos_day, sin_week, cos_week.

lags: lag_1-lag_6, lag_7, lag_14, lag_21, lag_28, lag_42, lag_56, lag_364, lag_365, rolling_mean_7, rolling_mean_14, rolling_mean_28, rolling_mean_364.

env_factors: dcoilwtico, dcoilwtico_ma7, dcoilwtico_ma28, onpromotion, onpromotion_ma7, onpromotion_ma28.

transactions: transactions_lag_16-transactions_lag_23. Эти лаги безопасны для test, потому что ссылаются только на train-период.

holidays: is_holiday_national, day_before_holiday, is_black_friday, is_cyber_monday, is_terremoto, is_navidad, is_dia_madre, is_futbol, is_dia_trabajo, is_primer_dia, is_dia_difuntos, work_day.

store: store_nbr, family, store_type, cluster.

**Features**


In [ ]:

full_idx = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max()),
        train["store_nbr"].unique(),
        train["family"].unique()
    ],
    names=["date", "store_nbr", "family"])

train = (
    train
    .set_index(["date", "store_nbr", "family"])
    .reindex(full_idx)
    .reset_index())

train["sales"] = train["sales"].fillna(0)
train["onpromotion"] = train["onpromotion"].fillna(0)

train_dates = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="D")

oil_dates = pd.Index(oil["date"].unique())
missing_oil_dates = train_dates.difference(oil_dates)


In [ ]:

print("Total days in train period:", len(train_dates))
print("Unique days in oil data:", oil["date"].nunique())
print("Missing oil days to reconstruct:", len(missing_oil_dates))

train_start = train["date"].min()
test_end = test["date"].max()


In [ ]:
oil = (
    oil
    .set_index("date")
    .reindex(pd.date_range(train_start, test_end, freq="D"))
    .rename_axis("date")
    .sort_index()
    .reset_index())

oil["dcoilwtico"] = oil["dcoilwtico"].interpolate(method="linear", limit_direction="both")
oil["dcoilwtico"] = oil["dcoilwtico"].round(2)



In [ ]:
store_sales = (
    train
    .groupby(["date", "store_nbr"])["sales"]
    .sum()
    .reset_index())

full_idx_stores = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max(), freq="D"),
        train["store_nbr"].unique()
    ],
    names=["date", "store_nbr"])

store_sales = (
    store_sales
    .set_index(["date", "store_nbr"])
    .reindex(full_idx_stores)
    .reset_index())

transactions = (
    transactions
    .merge(store_sales, on=["date", "store_nbr"], how="outer")
    .sort_values(["store_nbr", "date"]))

transactions.loc[transactions["sales"] == 0, "transactions"] = 0
transactions = transactions.drop(columns=["sales"])

transactions["transactions"] = (
    transactions
    .groupby("store_nbr")["transactions"]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both")))

In [ ]:
df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

df["day_of_week"]  = df["date"].dt.dayofweek
df["month"]        = df["date"].dt.month
df["year"]         = df["date"].dt.year
df["is_weekend"]   = (df["date"].dt.dayofweek >= 5).astype(int)
df["day_of_year"]  = df["date"].dt.dayofyear
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["date_index"]   = (df["date"] - df["date"].min()).dt.days

df["sin_day"]  = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["cos_day"]  = np.cos(2 * np.pi * df["day_of_year"] / 365)
df["sin_week"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["cos_week"] = np.cos(2 * np.pi * df["day_of_week"] / 7)



In [ ]:
for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]:
    df[f"lag_{lag}"] = df.groupby(["store_nbr", "family"])["sales"].shift(lag)

for window in [7, 14, 28]:
    df[f"rolling_mean_{window}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

df["rolling_mean_364"] = (
    df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(364, min_periods=30).mean()))



In [ ]:
df = df.merge(oil[["date", "dcoilwtico"]], on="date", how="left")

df = df.merge(
    stores[["store_nbr", "type", "cluster"]],
    on="store_nbr", how="left").rename(columns={"type": "store_type"})

df = df.merge(
    transactions[["date", "store_nbr", "transactions"]],
    on=["date", "store_nbr"], how="left")

for lag in range(16, 24):
    df[f"transactions_lag_{lag}"] = df.groupby("store_nbr")["transactions"].shift(lag)

df = df.drop(columns=["transactions"])

for col in ["dcoilwtico", "onpromotion"]:
    for w in [7, 28]:
        df[f"{col}_ma{w}"] = (
            df.groupby(["store_nbr", "family"])[col]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )



In [ ]:
national_hol = holidays[
    (holidays["locale"] == "National") & 
    (holidays["transferred"] == False)]["date"].unique()

df["is_holiday_national"] = df["date"].isin(national_hol).astype(int)
df["day_before_holiday"]  = (df["date"] + pd.Timedelta(days=1)).isin(national_hol).astype(int)

df["is_black_friday"] = df["date"].isin(
    holidays[holidays["description"] == "Black Friday"]["date"].unique()).astype(int)

df["is_cyber_monday"] = df["date"].isin(
    holidays[holidays["description"] == "Cyber Monday"]["date"].unique()).astype(int)

df["is_terremoto"] = df["date"].isin(
    holidays[holidays["description"].str.contains("Terremoto", na=False)]["date"].unique()).astype(int)

_hol = holidays[(holidays["locale"] == "National") & (holidays["transferred"] == False)]
df["is_navidad"]       = df["date"].isin(_hol[_hol["description"].str.contains("Navidad", na=False)]["date"].unique()).astype(int)
df["is_dia_madre"]     = df["date"].isin(_hol[_hol["description"].str.contains("Madre", na=False)]["date"].unique()).astype(int)
df["is_futbol"]        = df["date"].isin(_hol[_hol["description"].str.contains("futbol|F\xfatbol|Mundial", na=False)]["date"].unique()).astype(int)
df["is_dia_trabajo"]   = df["date"].isin(_hol[_hol["description"].str.contains("Trabajo", na=False)]["date"].unique()).astype(int)
df["is_primer_dia"]    = df["date"].isin(_hol[_hol["description"].str.contains("Primer", na=False)]["date"].unique()).astype(int)
df["is_dia_difuntos"]  = df["date"].isin(_hol[_hol["description"].str.contains("Difuntos", na=False)]["date"].unique()).astype(int)

work_day_dates = holidays[
    (holidays["type"] == "Work Day") | 
    (holidays["description"].str.contains("trabajo|Work", case=False, na=False) & (holidays["locale"] == "National"))]["date"].unique()
df["work_day"] = df["date"].isin(work_day_dates).astype(int)



In [ ]:
train_fe = df[df["sales"].notna()].copy()
test_fe  = df[df["sales"].isna()].copy()

train_fe = train_fe.drop(columns=["id", "day_name"], errors="ignore")
test_fe  = test_fe.drop(columns=["id", "day_name"], errors="ignore")

print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)

In [ ]:
FEATURES_V2 = [
    "day_of_week", "month", "year", "is_weekend", "day_of_year", "week_of_year", "date_index",
    "sin_day", "cos_day", "sin_week", "cos_week",
    "lag_1", "lag_2", "lag_3", "lag_4", "lag_5", "lag_6",
    "lag_7", "lag_14", "lag_21", "lag_28", "lag_42", "lag_56",
    "lag_364", "lag_365",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28", "rolling_mean_364",
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    "is_holiday_national", "day_before_holiday", "is_black_friday", "is_cyber_monday", 
    "is_terremoto", "is_navidad", "is_dia_madre", "is_futbol", "is_dia_trabajo", 
    "is_primer_dia", "is_dia_difuntos", "work_day",
    "store_nbr", "store_type", "cluster", "family"
]

# Features for encoding
CAT_FEATURES = ["store_nbr", "store_type", "cluster", "family"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

#to avoid changing the dataset
train_ensemble = train_fe.copy()
test_ensemble = test_fe.copy()

# 1. Encode categorical features
for col in CAT_FEATURES:
    le = LabelEncoder()
    train_ensemble[col] = le.fit_transform(train_ensemble[col].astype(str))
    test_ensemble[col] = le.transform(test_ensemble[col].astype(str))

#fill with 0
X = train_ensemble[FEATURES_V2].fillna(0)
y = np.log1p(train_ensemble["sales"]) # Log target for RMSLE optimization

X_test = test_ensemble[FEATURES_V2].fillna(0)

print(f"Features defined: {len(FEATURES_V2)}")
print(f"Training data shape: {X.shape}")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# ridge model
ridge = Ridge(alpha=1.0)
ridge.fit(X, y)


# random forest
# We use max_depth=10 to prevent overfitting on 3 million rows
rf = RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
rf.fit(X, y)


In [ ]:
from sklearn.metrics import mean_squared_log_error

SPLIT_DATE = "2017-08-01"

X_train_ens = X[train_ensemble["date"] < SPLIT_DATE]
y_train_ens = y[train_ensemble["date"] < SPLIT_DATE]

X_val_ens = X[train_ensemble["date"] >= SPLIT_DATE]
y_val_ens = y[train_ensemble["date"] >= SPLIT_DATE]

ridge_val = ridge.predict(X_val_ens).clip(0)
rf_val = rf.predict(X_val_ens).clip(0)


# (60% Ridge, 40% Forest)
final_val_pred = (0.6 * ridge_val) + (0.4 * rf_val)

score = np.sqrt(mean_squared_log_error(np.expm1(y_val_ens), np.expm1(final_val_pred)))
print(f"Ensemble RMSLE: {score:.5f}")

In [ ]:

SPLIT_DATE = pd.to_datetime("2017-08-01")


train_idx = train_fe[train_fe["date"] < SPLIT_DATE].index
val_idx = train_fe[train_fe["date"] >= SPLIT_DATE].index


cols_to_drop = ["date", "sales", "id"] 

X_train = train_fe.loc[train_idx].drop(columns=cols_to_drop, errors="ignore")
y_train = train_fe.loc[train_idx]["sales"]

X_val = train_fe.loc[val_idx].drop(columns=cols_to_drop, errors="ignore")
y_val = train_fe.loc[val_idx]["sales"]


print(f"Training on {len(X_train)} rows...")
ridge.fit(X_train, y_train)
rf.fit(X_train, y_train)


ridge_preds = ridge.predict(X_val).clip(0)
rf_preds = rf.predict(X_val).clip(0)
final_preds = (0.6 * ridge_preds) + (0.4 * rf_preds)

score = np.sqrt(mean_squared_log_error(np.expm1(y_val), np.expm1(final_preds)))
print(f"Realistic Validation RMSLE: {score:.5f}")

In [ ]:
final_val_pred = (0.8 * ridge_val) + (0.2 * rf_val)
score = np.sqrt(mean_squared_log_error(np.expm1(y_val_ens), np.expm1(final_val_pred)))
print(f"\nRealistic Validation RMSLE: {score:.5f}")

In [ ]:
# 1. Список колонок, которые точно нужно удалить
cols_to_drop = ['date', 'sales', 'id', 'day_name', 'day'] 

# 2. Оставляем в X только те колонки, которые являются числами (int или float)
# Это автоматически отфильтрует 'Tuesday', 'AUTOMOTIVE' и прочие строки
X = train_fe.drop(columns=[c for c in cols_to_drop if c in train_fe.columns])
X = X.select_dtypes(include=[np.number])

# 3. Теперь снова создаем выборки для ансамбля
X_train_ens = X[train_fe["date"] < SPLIT_DATE]
y_train_ens = y[train_fe["date"] < SPLIT_DATE]

X_val_ens = X[train_fe["date"] >= SPLIT_DATE]
y_val_ens = y[train_fe["date"] >= SPLIT_DATE]

# 4. Теперь обучение должно пройти без ошибок
print("Columns in X:", X.columns.tolist()) 
ridge.fit(X_train_ens, y_train_ens)
rf.fit(X_train_ens, y_train_ens)